# AutoGen Basic Sample 

In this code sample, you will use the [AutoGen](https://aka.ms/ai-agents/autogen) AI Framework to create a basic agent. 

The goal of this sample is to show you the steps that we will later use in the addtional code samples when implementing the different agentic patterns. 

## Import the Needed Python Packages 

In [2]:
import os
from dotenv import load_dotenv

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


## Create the Client 

In this sample, we will use [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) for access to the LLM. 

The `model` is defined as `gpt-4o-mini`. Try changing the model to another model available on the GitHub Models marketplace to see the different results. 

As a quick test, we will just run a simple prompt - `What is the capital of France`. 

In [3]:
load_dotenv()
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
    # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.getenv("GITHUB_TOKEN")),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)
import os
from azure.core.credentials import AzureKeyCredential
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.exceptions import ClientAuthenticationError

try:
    # Ensure the environment variable is set
    github_token = os.environ.get("GITHUB_TOKEN")
    if not github_token:
        raise ValueError("GITHUB_TOKEN environment variable is not set")


    client = AzureAIChatCompletionClient(
        model="gpt-4o-mini",
        endpoint="https://models.inference.ai.azure.com",
         # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
         # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
         credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
             model_info={
                "json_output": True,
                "function_calling": True,
                 "vision": True,
                "family": "unknown",
            },  
    )

    result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
    print(result)


except ClientAuthenticationError as e:
    print(f"ClientAuthenticationError: {e}")
except ValueError as e:
    print(f"ValueError: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

finish_reason='stop' content='The capital of France is Paris.' usage=RequestUsage(prompt_tokens=14, completion_tokens=7) cached=False logprobs=None thought=None


## Defining the Agent 

Now that we have set up the `client` and confirmed that it is working, let us create an `AssistantAgent`. Each agent can be assigned a: 
**name** - A short hand name that will be useful in referencing it in multi-agent flows. 
**model_client** - The client that you created in the earlier step. 
**tools** - Available tools that the Agent can use to complete a task.
**system_message** - The metaprompt that defines the task, behavior and tone of the LLM. 

You can change the system message to see how the LLM responds. We will cover `tools` in Lesson #4. 


In [17]:
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[],
    system_message="You are a travel agent that plans great vacations",
)

## Run the Agent 

The below function will run the agent. We use the the `on_message` method to update the Agent's state with the new message. 

In this case, we update the state with a new message from the user which is `"Plan me a great sunny vacation"`.

You can change the message content to see how the LLM responds differently. 

In [18]:
async def assistant_run() -> None:
    response = await agent.on_messages(
        [TextMessage(content="Plan me a great sunny vacation", source="user")],
        cancellation_token=CancellationToken(),
    )
    print(response.inner_messages)
    print(response.chat_message)


# Use asyncio.run(assistant_run()) when running in a script.
await assistant_run()

ClientAuthenticationError: (unauthorized) Bad credentials
Code: unauthorized
Message: Bad credentials